# Thuật toán tìm kiếm theo chiều rộng theo 2 cách:
- Cách 1: Khi tìm thấy GOAL (con sinh từ node đang xét) không dừng vòng lặp tiếp tục chạy hết các node trong Frontier cho đến khi thấy goal
- Cách 2: Khi tìm thấy GOAL (con sinh từ node đang xét) dừng ngay và trả về node đó

# Project GitHub link (Repo lưu toàn bộ bài tập của em)

https://github.com/quangbru376-blip/Lab-AI

### BREADTH-FIRST-SEARCH - Cách 1
function BREADTH-FIRST-SEARCH(problem) returns a solution or failure  
    node ← NODE(problem.INITIAL)  
    if problem.GOAL-TEST(node.STATE) then  
        return SOLUTION(node)  

    frontier ← FIFO-QUEUE()    # danh sách node chờ mở rộng  
    frontier.INSERT(node)  

    reached ← ∅    # tập các state đã khám phá  

    while not EMPTY?(frontier) do  
        node ← frontier.REMOVE()    # lấy node đầu tiên trong queue  
        reached ← reached ∪ {node.STATE}  

        if problem.GOAL-TEST(node.STATE) then  
            return SOLUTION(node)  

        for each action in problem.ACTIONS(node.STATE) do  
            child ← CHILD_NODE(problem, node, action)  

            if child.STATE ∉ reached ∧ child ∉ frontier then  
                frontier.INSERT(child)  

    return failure

In [ ]:
import random, copy

class Node():
    def __init__(self, floor_state , position: tuple, parent, birth_action, level):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.level = level
    
class Queue():
    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0

    def enqueue(self, node):
        if (node):
            self.queue.append(node)
        else:
            raise Exception("Empty node Error")

    def peek(self):
        if self.is_empty():
            return None
        return self.queue[0]
    
    def dequeue(self):
        if self.is_empty():
            raise Exception("Empty queue Error")
        return self.queue.pop(0)
        
    def contain(self, other: Node):
        for node in self.queue:
            if (node.floor_state == other.floor_state and node.position == other.position):
                return True
        return False
        
#####


GOAL_STATE = [[0, 0, 0],
              [0, 0, 0],
              [0, 0, 0]]
ROW = 3
COL = 3

def floor_and_vacuumpos_initialize(row, col):
    """
    This function generate the floor with random dirty tiles, vacumm with random position
    """
    floor = []
    vacuum_pos = (random.randint(0, row - 1), random.randint(0, col - 1))
    for i in range(row):
        floor.append([random.randint(0, 1) for j in range(col)])
        
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def posible_moves(vacuum_pos):
    """
    This function calculate the possible moves of the vacuum
    """
    moves = []
    if vacuum_pos[0] > 0:
        moves.append("UP")
    if vacuum_pos[0] < ROW - 1:
        moves.append("DOWN")
    if vacuum_pos[1] > 0:
        moves.append("LEFT")
    if vacuum_pos[1] < COL - 1:
        moves.append("RIGHT")
    return moves

def apply_move(floor, vacuum_pos, move):
    """
    This function apply the move 
    """
    temp_floor = copy.deepcopy(floor)
    temp_vac_pos = vacuum_pos
    if move == "UP":
        temp_vac_pos = (temp_vac_pos[0] - 1, temp_vac_pos[1])
    elif move == "DOWN":
        temp_vac_pos = (temp_vac_pos[0] + 1, temp_vac_pos[1])
    elif move == "LEFT":
        temp_vac_pos = (temp_vac_pos[0], temp_vac_pos[1] - 1)
    elif move == "RIGHT":
        temp_vac_pos = (temp_vac_pos[0], temp_vac_pos[1] + 1)
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
        
    return temp_floor, temp_vac_pos

def path(goal_node):
    """
    This function find the way from root to goal node and print it
    """
    current = goal_node
    path = []
    
    while current.parent is not None:
        path.append(current)
        current = current.parent
    path.append(current)
    path.reverse() #from goal-> root to root -> goal
    
    for i, node in enumerate(path):
        node.floor_state[node.position[0]][node.position[1]] = "x"
        if i == 0:
            print(f"Initial floor:")
        else:    
            print(f"Node: {i}")
            print(f"Floor state:")
        for row in node.floor_state:
            print(row)
        print(f"Vacuum Position: {node.position}")
        print(f"Birth move: {node.birth_action}")
        print(f"=" * 20)
        
        
def main():
    floor, vacuum_pos = floor_and_vacuumpos_initialize(ROW, COL)
    #floor =[[0, 1, 0], [1, 0, 1], [0, 0, 1]]
    #vacuum_pos = (2, 1)
    frontier = Queue()
    goal_reached = False
    step = 0
    goal_node = None

    visited_state = []
    root = Node(floor, vacuum_pos, None, None, 0)
    frontier.enqueue(root)

    while True:
        if frontier.is_empty(): #dừng khi frontier trống (hết cách)
            print("Frontier is empty!!")
            break
        
        current_node = frontier.dequeue()
        step += 1
        if (current_node.floor_state, current_node.position) not in visited_state: #check nếu state node có trong visited
            visited_state.append((current_node.floor_state, current_node.position))
        else:
            continue
        if current_node.floor_state == GOAL_STATE: #dừng khi tìm thấy đáp án (GOAL)
            goal_node = current_node
            goal_reached = True
            break
        
        moves = posible_moves(current_node.position)
        for move in moves:# chạy thử từng bước có thể để sinh node con
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move) 
            temp_node = Node(temp_floor, temp_vacuum_pos, current_node, move, current_node.level + 1) #sinh node con

            if not frontier.contain(temp_node) and (temp_floor, temp_vacuum_pos) not in visited_state: 
                #kiểm tra nếu node con không có trong visited hoặc frontier
                frontier.enqueue(temp_node)
        
        if goal_reached:
            break
    
    print(f">>> Program end.")
    print(f">>> Goal found. The whole floor has been cleaned")
    print(f">>> {step} steps has been through")
    path(goal_node) 
    
        
        
if __name__ == "__main__":
    main()

### BREADTH-FIRST-SEARCH - Cách 2
function BREADTH-FIRST-SEARCH(problem) returns a solution or failure  
    node ← NODE(problem.INITIAL)  
    if problem.GOAL-TEST(node.STATE) then  
        return SOLUTION(node)  

    frontier ← FIFO-QUEUE()    # danh sách node chờ mở rộng  
    frontier.INSERT(node)  

    reached ← ∅    # tập các state đã khám phá  

    while not EMPTY?(frontier) do  
        node ← frontier.REMOVE()    # lấy node đầu tiên trong queue  
        reached ← reached ∪ {node.STATE}  

        if problem.GOAL-TEST(node.STATE) then  
            return SOLUTION(node)  

        for each action in problem.ACTIONS(node.STATE) do  
            child ← CHILD_NODE(problem, node, action)  

            if child.STATE ∉ reached ∧ child ∉ frontier then  
                if problem.GOAL-TEST(child.STATE)
                    return SOLUTION(child)
                frontier.INSERT(child)  

    return failure

In [ ]:
import random, copy

class Node():
    def __init__(self, floor_state , position: tuple, parent, birth_action, level):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.level = level
    
class Queue():
    def __init__(self):
        self.queue = []

    def is_empty(self):
        return len(self.queue) == 0

    def enqueue(self, node):
        if (node):
            self.queue.append(node)
        else:
            raise Exception("Empty node Error")

    def peek(self):
        if self.is_empty():
            return None
        return self.queue[0]
    
    def dequeue(self):
        if self.is_empty():
            raise Exception("Empty queue Error")
        return self.queue.pop(0)
        
    def contain(self, other: Node):
        for node in self.queue:
            if (node.floor_state == other.floor_state and node.position == other.position):
                return True
        return False
        
#####


GOAL_STATE = [[0, 0, 0],
              [0, 0, 0],
              [0, 0, 0]]
ROW = 3
COL = 3

def floor_and_vacuumpos_initialize(row, col):
    """
    This function generate the floor with random dirty tiles, vacumm with random position
    """
    floor = []
    vacuum_pos = (random.randint(0, row - 1), random.randint(0, col - 1))
    for i in range(row):
        floor.append([random.randint(0, 1) for j in range(col)])
        
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def posible_moves(vacuum_pos):
    """
    This function calculate the possible moves of the vacuum
    """
    moves = []
    if vacuum_pos[0] > 0:
        moves.append("UP")
    if vacuum_pos[0] < ROW - 1:
        moves.append("DOWN")
    if vacuum_pos[1] > 0:
        moves.append("LEFT")
    if vacuum_pos[1] < COL - 1:
        moves.append("RIGHT")
    return moves

def apply_move(floor, vacuum_pos, move):
    """
    This function apply the move 
    """
    temp_floor = copy.deepcopy(floor)
    temp_vac_pos = vacuum_pos
    if move == "UP":
        temp_vac_pos = (temp_vac_pos[0] - 1, temp_vac_pos[1])
    elif move == "DOWN":
        temp_vac_pos = (temp_vac_pos[0] + 1, temp_vac_pos[1])
    elif move == "LEFT":
        temp_vac_pos = (temp_vac_pos[0], temp_vac_pos[1] - 1)
    elif move == "RIGHT":
        temp_vac_pos = (temp_vac_pos[0], temp_vac_pos[1] + 1)
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
        
    return temp_floor, temp_vac_pos

def path(goal_node):
    """
    This function find the way from root to goal node and print it
    """
    current = goal_node
    path = []
    
    while current.parent is not None:
        path.append(current)
        current = current.parent
    path.append(current)
    path.reverse() #from goal-> root to root -> goal
    
    for i, node in enumerate(path):
        node.floor_state[node.position[0]][node.position[1]] = "x"
        if i == 0:
            print(f"Initial floor:")
        else:    
            print(f"Node: {i}")
            print(f"Floor state:")
        for row in node.floor_state:
            print(row)
        print(f"Vacuum Position: {node.position}")
        print(f"Birth move: {node.birth_action}")
        print(f"=" * 20)
        
        
def main():
    floor, vacuum_pos = floor_and_vacuumpos_initialize(ROW, COL)
    #floor =[[0, 1, 0], [1, 0, 1], [0, 0, 1]]
    #vacuum_pos = (2, 1)
    frontier = Queue()
    goal_reached = False
    step = 0
    goal_node = None

    visited_state = []
    root = Node(floor, vacuum_pos, None, None, 0)
    frontier.enqueue(root)

    while True:
        if frontier.is_empty(): #dừng khi frontier trống (hết cách)
            print("Frontier is empty!!")
            break
        
        current_node = frontier.dequeue()
        step += 1
        if (current_node.floor_state, current_node.position) not in visited_state: #check nếu state node có trong visited
            visited_state.append((current_node.floor_state, current_node.position))
        else:
            continue
        if current_node.floor_state == GOAL_STATE: #dừng khi tìm thấy đáp án (GOAL)
            goal_node = current_node
            goal_reached = True
            break
        
        moves = posible_moves(current_node.position)
        for move in moves:# chạy thử từng bước có thể để sinh node con
            temp_floor, temp_vacuum_pos = apply_move(current_node.floor_state, current_node.position, move) 
            temp_node = Node(temp_floor, temp_vacuum_pos, current_node, move, current_node.level + 1) #sinh node con
            if temp_floor == GOAL_STATE: #điểm khác cách 1
                goal_node = temp_node
                goal_reached = True
                break
            if not frontier.contain(temp_node) and (temp_floor, temp_vacuum_pos) not in visited_state: 
                #kiểm tra nếu node con không có trong visited hoặc frontier
                frontier.enqueue(temp_node)
        
        if goal_reached:
            break
    
    print(f">>> Program end.")
    print(f">>> Goal found. The whole floor has been cleaned")
    print(f">>> {step} steps has been through")
    path(goal_node) 
    
        
        
if __name__ == "__main__":
    main()